# Chapter 26 — Securitization, Structured Credit, and the Crisis

Every number in the chapter's six examples, from the loss distribution that makes a senior tranche convex in a parameter to the overcollateralization test that deleverages a CLO on its own.

| Section | What it reproduces | Example |
|---|---|---|
| 1 | The tranche loss function, the copula's loss distribution, and what a tranche does to a parameter error | 26.1 |
| 2 | Negative convexity of a pass-through, measured against a seven-year bullet | 26.2 |
| 3 | Where a bond's credit support comes from, and where it goes under stress | 26.3 |
| 4 | A CDO of mezzanine tranches: the copula applied twice | 26.4 |
| 5 | The overcollateralization test | 26.5 |
| 6 | Why a senior claim recovers less without the borrower getting worse | 26.6 |

All of it is deterministic — there is no simulation and no seed. The copula's loss
distribution is integrated over the common factor on a fine grid rather than sampled,
so every run returns the same digits, and Section 4 builds the re-securitization's loss
distribution by exact convolution instead of by Monte Carlo.

Column headers and the labels passed to `show()` stay in English: CJK characters are
double-width in a monospace terminal while Python's `%-50s` counts characters, so
translating one would misalign every row beneath it.

## Setup

Two functions carry the chapter. `loss_pmf` is equation (24.13): the number of defaults
in a pool of $n$ loans under the one-factor Gaussian copula of Section 24.13,
$X_i = \sqrt{\rho}\,M + \sqrt{1-\rho}\,Z_i$, integrated over the common factor $M$.
`tranche_loss` is equation (25.8), the call spread on pool loss that defines a tranche;
`tranche_el` averages it over the loss distribution. `show()` prints each computed value
beside the one the chapter prints.

In [1]:
import numpy as np
from math import sqrt
from scipy.stats import norm, binom

N = norm.cdf


def show(label, got, printed, tol=6e-4):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-50s %12.4f   book %11.4f   %s" % (label, got, printed, flag))


def loss_pmf(n, pd, rho, lim=12.0, npts=24001):
    """P(K = k defaults) under the one-factor Gaussian copula -- equation (24.13).

    Conditional on the common factor M each of the n loans defaults with
    probability p(M) = N((N^-1(pd) - sqrt(rho) M) / sqrt(1 - rho)), independently,
    so K | M is binomial; the unconditional law integrates the binomial against the
    density of M. Simpson's rule on [-lim, lim]: at high correlation p(M) sweeps the
    whole unit interval as M moves, and a truncated range understates every tranche.
    """
    m = np.linspace(-lim, lim, npts)
    w = norm.pdf(m) * (2 * lim / (npts - 1)) / 3.0
    w[1:-1:2] *= 4.0
    w[2:-1:2] *= 2.0
    p_m = N((norm.ppf(pd) - sqrt(rho) * m) / sqrt(1.0 - rho))
    k = np.arange(n + 1)
    return w @ binom.pmf(k[None, :], n, p_m[:, None])


def tranche_loss(L, a, b):
    """Equation (25.8): the a-b tranche's loss per unit of its notional at pool loss L."""
    return np.clip(L - a, 0.0, b - a) / (b - a)


def tranche_el(pmf, n, lgd, a, b):
    """Expected loss of the a-b tranche: (25.8) averaged over the loss distribution."""
    L = lgd * np.arange(n + 1) / n
    return float(pmf @ tranche_loss(L, a, b))

## 1. What a tranche does to a parameter error — Example 26.1

First the left panel of Figure 26.2: a tranche's loss is a call spread on the pool's,
zero until the attachment point and total at the detachment point. Then the table of
Example 26.1 — 100 loans, 60% loss given default, correlation 0.15 — where the pool's
expected loss is linear in the default probability and the 12–30% senior tranche's is
not. **Tripling the default probability triples the pool's loss and multiplies the
senior tranche's by twenty-eight.**

In [2]:
for L in (0.0, 0.03, 0.05, 0.07, 0.12, 0.21, 0.30, 0.35):
    print("pool loss %4.0f%%   equity %6.1f%%   mezzanine %6.1f%%   senior %6.1f%%"
          % (100 * L, 100 * tranche_loss(L, 0.00, 0.03), 100 * tranche_loss(L, 0.03, 0.07),
             100 * tranche_loss(L, 0.12, 0.30)))
print()

n, lgd, rho = 100, 0.60, 0.15
print("%-24s %18s %26s" % ("true default probability", "pool expected loss", "senior 12-30% expected loss"))
book = {0.05: (3.00, 0.266), 0.06: (3.60, 0.486), 0.08: (4.80, 1.208),
        0.10: (6.00, 2.369), 0.15: (9.00, 7.374), 0.20: (12.00, 15.205)}
senior = {}
for pd in sorted(book):
    pmf = loss_pmf(n, pd, rho)
    pool = 100 * lgd * pd                      # linear in the parameter
    senior[pd] = 100 * tranche_el(pmf, n, lgd, 0.12, 0.30)
    print("%22.0f%% %17.2f%% %25.3f%%" % (100 * pd, pool, senior[pd]))
print()
show("senior expected loss at 5%, %", senior[0.05], 0.266)
show("senior expected loss at 15%, %", senior[0.15], 7.374)
show("tripling the default probability multiplies it by", senior[0.15] / senior[0.05], 28.0, tol=0.02)

pool loss    0%   equity    0.0%   mezzanine    0.0%   senior    0.0%
pool loss    3%   equity  100.0%   mezzanine    0.0%   senior    0.0%
pool loss    5%   equity  100.0%   mezzanine   50.0%   senior    0.0%
pool loss    7%   equity  100.0%   mezzanine  100.0%   senior    0.0%
pool loss   12%   equity  100.0%   mezzanine  100.0%   senior    0.0%
pool loss   21%   equity  100.0%   mezzanine  100.0%   senior   50.0%
pool loss   30%   equity  100.0%   mezzanine  100.0%   senior  100.0%
pool loss   35%   equity  100.0%   mezzanine  100.0%   senior  100.0%

true default probability pool expected loss senior 12-30% expected loss


                     5%              3.00%                     0.266%
                     6%              3.60%                     0.486%


                     8%              4.80%                     1.208%
                    10%              6.00%                     2.369%


                    15%              9.00%                     7.374%


                    20%             12.00%                    15.205%

senior expected loss at 5%, %                            0.2664   book      0.2660   ok
senior expected loss at 15%, %                           7.3738   book      7.3740   ok
tripling the default probability multiplies it by       27.6772   book     28.0000   ok


**One row is worth working in full.** Each default costs $0.6$ of a $1/100$ share, so
the pool's loss is $0.006K$ after $K$ defaults; the 12–30% tranche is untouched until
$K \ge 21$ and exhausted once $K \ge 50$, and between the two it loses
$(0.006K - 0.12)/0.18$ of its notional. The tranche's expected loss is that function summed
against $\mathbb{P}(K)$ — and the number that moves is the probability of *reaching* the
attachment point, 1.50% at a 5% default probability and 24.8% at 15%.

In [3]:
k = np.arange(n + 1)
L = lgd * k / n                                # 0.006 K
first_hit = int(k[L > 0.12][0])
exhausted = int(k[L >= 0.30][0])
print("the 12-30%% tranche is untouched until K = %d and exhausted from K = %d" % (first_hit, exhausted))
print("between the two it loses (0.006 K - 0.12) / 0.18 of its notional")
print()
for pd, p_book, el_book in ((0.05, 1.50, 0.266), (0.15, 24.8, 7.374)):
    pmf = loss_pmf(n, pd, rho)
    p_reach = 100 * pmf[first_hit:].sum()
    el = 100 * float(pmf[first_hit:] @ ((0.006 * k[first_hit:] - 0.12) / 0.18).clip(max=1.0))
    show("PD = %2.0f%%: P(K >= 21), %%" % (100 * pd), p_reach, p_book, tol=0.01)
    show("PD = %2.0f%%: sum of P(K) x tranche loss, %%" % (100 * pd), el, el_book)

the 12-30% tranche is untouched until K = 21 and exhausted from K = 50
between the two it loses (0.006 K - 0.12) / 0.18 of its notional



PD =  5%: P(K >= 21), %                                  1.5022   book      1.5000   ok
PD =  5%: sum of P(K) x tranche loss, %                  0.2664   book      0.2660   ok
PD = 15%: P(K >= 21), %                                 24.7536   book     24.8000   ok
PD = 15%: sum of P(K) x tranche loss, %                  7.3738   book      7.3740   ok


## 2. Negative convexity, measured — Example 26.2

A 30-year 5.5% pass-through on a level-pay pool, paying monthly, at a 5% yield
compounded monthly. The prepayment speed is a CPR that responds to the refinancing
incentive — 35 / 22 / 12 / 8 / 6% at $\Delta y = -2, \ldots, +2$ percent — applied
monthly as $1 - (1-\text{CPR})^{1/12}$ of the balance left after scheduled principal.
The comparison is a seven-year 5.5% semiannual bullet at a semiannually compounded yield, seven years being about the pool's
weighted average life at the base speed. The "fixed speed" column keeps 12% at every
yield, so the difference between the first two columns is the prepayment option alone.

In [4]:
COUPON, Y0, TERM = 0.055, 0.05, 30
CPR = {-2: 0.35, -1: 0.22, 0: 0.12, 1: 0.08, 2: 0.06}   # prepayment speed by yield change


def pool_cash_flows(cpr, coupon=COUPON, years=TERM):
    """Monthly cash flows of a level-pay pool at a constant CPR, per unit of face."""
    n, r = 12 * years, coupon / 12.0
    smm = 1.0 - (1.0 - cpr) ** (1.0 / 12.0)
    bal, cf, principal = 1.0, [], []
    for m in range(1, n + 1):
        payment = bal * r / (1.0 - (1.0 + r) ** -(n - m + 1))   # level payment on what is left
        interest = bal * r
        sched = payment - interest
        prepay = (bal - sched) * smm
        cf.append(interest + sched + prepay)
        principal.append(sched + prepay)
        bal -= sched + prepay
    return np.array(cf), np.array(principal)


def passthrough(y, cpr):
    cf, _ = pool_cash_flows(cpr)
    m = np.arange(1, len(cf) + 1)
    return float(cf @ (1.0 + y / 12.0) ** -m)


def bullet(y, coupon=COUPON, years=7):
    k = np.arange(1, 2 * years + 1)
    return float((coupon / 2.0) * ((1.0 + y / 2.0) ** -k).sum() + (1.0 + y / 2.0) ** -(2 * years))


def wal(cpr):
    _, principal = pool_cash_flows(cpr)
    return float(principal @ np.arange(1, len(principal) + 1)) / 12.0


book = {-2: (1.0534, 1.1399, 1.1568), -1: (1.0496, 1.0797, 1.0908), 0: (1.0253, 1.0253, 1.0292),
        1: (0.9698, 0.9759, 0.9718), 2: (0.9035, 0.9307, 0.9181)}
print("%6s %6s %14s %16s %12s" % ("dy", "CPR", "pass-through", "fixed 12% CPR", "7y bullet"))
price = {}
for dy in sorted(CPR):
    y = Y0 + dy / 100.0
    price[dy] = (passthrough(y, CPR[dy]), passthrough(y, CPR[0]), bullet(y))
    print("%+5d%% %5.0f%% %14.4f %16.4f %12.4f" % (dy, 100 * CPR[dy], *price[dy]))
print()
for dy in sorted(book):
    for j, name in enumerate(("pass-through", "fixed speed", "bullet")):
        show("dy = %+d%%: %s" % (dy, name), price[dy][j], book[dy][j], tol=6e-5)
print()
show("pool weighted average life at 12% CPR, years", wal(CPR[0]), 6.6, tol=0.01)

    dy    CPR   pass-through    fixed 12% CPR    7y bullet
   -2%    35%         1.0534           1.1399       1.1568
   -1%    22%         1.0496           1.0797       1.0908
   +0%    12%         1.0253           1.0253       1.0292
   +1%     8%         0.9698           0.9759       0.9718
   +2%     6%         0.9035           0.9307       0.9181

dy = -2%: pass-through                                   1.0534   book      1.0534   ok
dy = -2%: fixed speed                                    1.1399   book      1.1399   ok
dy = -2%: bullet                                         1.1568   book      1.1568   ok
dy = -1%: pass-through                                   1.0496   book      1.0496   ok
dy = -1%: fixed speed                                    1.0797   book      1.0797   ok
dy = -1%: bullet                                         1.0908   book      1.0908   ok
dy = +0%: pass-through                                   1.0253   book      1.0253   ok
dy = +0%: fixed speed        

Effective duration and convexity from the $\pm 1\%$ prices,
$D = (P_- - P_+)/(2P_0\,\Delta y)$ and $C = (P_- + P_+ - 2P_0)/(P_0\,\Delta y^2)$. The
responding pass-through is the only instrument with negative convexity, and the roughly
350-point gap between the two pass-through convexities is the option, measured. In a 200 bp
rally the bullet gains 12.4% and the pass-through 2.7%; in a 200 bp sell-off they lose about
the same.

In [5]:
def duration_convexity(p_down, p_0, p_up, dy=0.01):
    """Effective duration and convexity from the -dy / 0 / +dy prices."""
    return (p_down - p_up) / (2.0 * p_0 * dy), (p_down + p_up - 2.0 * p_0) / (p_0 * dy ** 2)


print("%-28s %10s %11s" % ("instrument", "duration", "convexity"))
for j, (name, d_book, c_book) in enumerate((("pass-through, responding", 3.89, -304.0),
                                            ("pass-through, fixed 12% CPR", 5.07, 48.5),
                                            ("seven-year bullet", 5.78, 39.9))):
    d, c = duration_convexity(price[-1][j], price[0][j], price[1][j])
    print("%-28s %10.2f %11.1f     book %5.2f %7.1f" % (name, d, c, d_book, c_book))
print()
print("a 2% move, in percent of the starting price:")
show("rally: bullet", 100 * (price[-2][2] / price[0][2] - 1), 12.4, tol=0.02)
show("rally: pass-through, speed 12% -> 35%", 100 * (price[-2][0] / price[0][0] - 1), 2.7, tol=0.02)
show("sell-off: bullet", 100 * (price[2][2] / price[0][2] - 1), -10.8, tol=0.02)
show("sell-off: pass-through, speed 12% -> 6%", 100 * (price[2][0] / price[0][0] - 1), -11.9, tol=0.02)

instrument                     duration   convexity
pass-through, responding           3.89      -304.5     book  3.89  -304.0
pass-through, fixed 12% CPR        5.07        48.5     book  5.07    48.5
seven-year bullet                  5.78        39.9     book  5.78    39.9

a 2% move, in percent of the starting price:
rally: bullet                                           12.3942   book     12.4000   ok
rally: pass-through, speed 12% -> 35%                    2.7418   book      2.7000   ok
sell-off: bullet                                       -10.7975   book    -10.8000   ok
sell-off: pass-through, speed 12% -> 6%                -11.8789   book    -11.9000   ok


## 3. Where the protection comes from — Example 26.3

A \$1,000 million pool at an 8.5% coupon backs bonds costing 5.5% of the pool; fees take 0.5%, so the
excess spread is 2.5% a year of the performing balance. A senior bond attaching at 20%
starts with 22% of support — 20% subordination plus 2% overcollateralization — and the
promised column adds 2.5% a year. Under stress, 8% of the original balance defaults each
year at a 40% recovery: the pool loses 4.8% a year, the performing balance falls by the
full 8%, and the spread is earned only on the average balance still performing. Every
figure is a percentage of the original \$1,000 million. Losses hit overcollateralization
and spread earned before they touch subordination.

In [6]:
spread = 8.5 - 5.5 - 0.5          # excess spread, % a year of the performing balance
sub, oc = 20.0, 2.0                # subordination and overcollateralization, % of the original pool
default_rate, recovery = 0.08, 0.40
loss = 100 * default_rate * (1 - recovery)   # 4.8% of the original pool a year

show("excess spread, % a year", spread, 2.5)
show("opening support = subordination + OC, %", sub + oc, 22.0)
print()
print("%4s %10s %14s %8s %9s %10s" % ("year", "promised", "spread earned", "loss", "OC left", "support"))
support, cushion, earned = sub + oc, oc, []
book = {1: (24.5, 2.40, 19.60), 2: (27.0, 2.20, 17.00), 3: (29.5, 2.00, 14.20),
        4: (32.0, 1.80, 11.20), 5: (34.5, 1.60, 8.00)}
for t in range(1, 6):
    promised = sub + oc + spread * t
    performing_avg = 1.0 - default_rate * (t - 0.5)      # 96%, 88%, 80%, 72%, 64%
    sp = spread * performing_avg
    earned.append(sp)
    cushion = max(cushion + sp - loss, 0.0)               # OC and spread absorb losses first
    support += sp - loss
    print("%4d %9.1f%% %13.2f%% %7.2f%% %8.2f%% %9.2f%%" % (t, promised, sp, loss, cushion, support))
    for got, want, name in zip((promised, sp, support), book[t], ("promised", "spread earned", "support")):
        assert abs(got - want) < 1e-9, (t, name)
print()
print("year 1 in full: 22.0 + %.2f - %.2f = %.2f;  OC 2 + spread %.2f absorb %.2f, so %.2f comes out of subordination"
      % (earned[0], loss, 22.0 + earned[0] - loss, earned[0], 2 + earned[0], loss - 2 - earned[0]))
print("year 2 in full: %.2f - (%.2f - %.2f) = %.2f = 22 - 2 - 0.40 - 2.60"
      % (22.0 + earned[0] - loss, loss, earned[1], 22.0 + earned[0] + earned[1] - 2 * loss))
print()
show("swing from the promised 34.5%, points", 34.5 - support, 26.5)
show("cumulative pool loss, %", 5 * loss, 24.0)
show("spread earned over five years, %", sum(earned), 10.0)
show("subordination consumed, of 20, %", 5 * loss - oc - sum(earned), 12.0)

excess spread, % a year                                  2.5000   book      2.5000   ok
opening support = subordination + OC, %                 22.0000   book     22.0000   ok

year   promised  spread earned     loss   OC left    support
   1      24.5%          2.40%    4.80%     0.00%     19.60%
   2      27.0%          2.20%    4.80%     0.00%     17.00%
   3      29.5%          2.00%    4.80%     0.00%     14.20%
   4      32.0%          1.80%    4.80%     0.00%     11.20%
   5      34.5%          1.60%    4.80%     0.00%      8.00%

year 1 in full: 22.0 + 2.40 - 4.80 = 19.60;  OC 2 + spread 2.40 absorb 4.40, so 0.40 comes out of subordination
year 2 in full: 19.60 - (4.80 - 2.20) = 17.00 = 22 - 2 - 0.40 - 2.60

swing from the promised 34.5%, points                   26.5000   book     26.5000   ok
cumulative pool loss, %                                 24.0000   book     24.0000   ok
spread earned over five years, %                        10.0000   book     10.0000   ok
subordinat

## 4. A CDO of mezzanine tranches — Example 26.4

First the tranches of Example 26.1's pool at $\rho = 0.15$: the 3–7% mezzanine piece
loses 20.19% in expectation, and the width-weighted losses add to the pool's 3.00%.

In [7]:
pmf = loss_pmf(n, 0.05, rho)
layers = ((0.00, 0.03, "equity 0-3%", 64.07), (0.03, 0.07, "mezzanine 3-7%", 20.19),
          (0.07, 0.12, "mezzanine 7-12%", 4.45), (0.12, 0.30, "senior 12-30%", 0.27),
          (0.30, 1.00, "super senior 30-100%", 0.00))
for a, b, name, want in layers:
    show(name, 100 * tranche_el(pmf, n, lgd, a, b), want, tol=6e-3)
show("width-weighted sum = pool expected loss, %",
     100 * sum((b - a) * tranche_el(pmf, n, lgd, a, b) for a, b, _, _ in layers), 3.00)

equity 0-3%                                             64.0656   book     64.0700   ok
mezzanine 3-7%                                          20.1870   book     20.1900   ok
mezzanine 7-12%                                          4.4508   book      4.4500   ok
senior 12-30%                                            0.2664   book      0.2700   ok
super senior 30-100%                                     0.0001   book      0.0000   ok
width-weighted sum = pool expected loss, %               3.0000   book      3.0000   ok


Now pool the 3–7% piece from one hundred different deals and tranche the result. In
Section 24.13's notation, loan $j$ of deal $i$ has $X_{ij} = \sqrt{0.15}\,M_i + \sqrt{0.85}\,Z_{ij}$
and the deal factors load on one market factor, $M_i = \sqrt{\rho}\,M + \sqrt{1-\rho}\,Z_i$,
with $\rho$ the inter-deal correlation. Conditional on $M$ the hundred pieces are
independent, so the new pool's loss distribution is Step 2 of Section 25.7 with a
piece's loss distribution in the place a single default occupied — here an exact
100-fold convolution on a grid of 0.05 of a piece, since a 3–7% layer of a 100-loan pool
can only lose $0, 0.15, \ldots, 0.90$ or $1$ of itself.

**The senior layer of the re-securitization goes from losing 1.08% to 11.25% of its notional
as the inter-deal correlation rises from 0.10 to 0.90**, and every row adds back to 20.19%,
as it must.

In [8]:
RHO_LOAN, N_DEALS = 0.15, 100
UNITS = 0.05                                   # a 3-7% piece of a 100-loan pool loses 0, 0.15, ..., 0.90 or 1
m_grid = np.arange(-8.0, 8.0 + 1e-9, 0.02)
k = np.arange(n + 1)
p_loan = N((norm.ppf(0.05) - sqrt(RHO_LOAN) * m_grid) / sqrt(1.0 - RHO_LOAN))   # p(M_i) for each deal factor
count_pmf_given_deal = binom.pmf(k[None, :], n, p_loan[:, None])              # K_i | M_i, one row per M_i
piece_units = np.rint(tranche_loss(lgd * k / n, 0.03, 0.07) / UNITS).astype(int)  # piece loss at each K


def piece_dist_given_market(market, rho_deal):
    """Loss distribution of one 3-7% piece given the market factor M, on the 0.05 grid.

    M_i = sqrt(rho) M + sqrt(1 - rho) Z_i, so given M the deal factor is normal
    with mean sqrt(rho) M and standard deviation sqrt(1 - rho); integrate K_i | M_i
    over that law and map each count to the piece's loss.
    """
    w = norm.pdf(m_grid, loc=sqrt(rho_deal) * market, scale=sqrt(1.0 - rho_deal)) * 0.02
    pk = w @ count_pmf_given_deal
    d = np.bincount(piece_units, weights=pk, minlength=21)
    return d / d.sum()


def convolve_power(d, times):
    """The law of a sum of `times` independent copies of d, by repeated squaring."""
    result, base = np.array([1.0]), d
    while times:
        if times & 1:
            result = np.convolve(result, base)
        times >>= 1
        if times:
            base = np.convolve(base, base)
    return result


def cdo_squared(rho_deal, layers=((0.30, 1.00), (0.12, 0.30), (0.03, 0.12), (0.00, 0.03)), h=0.125, lim=7.0):
    """Expected loss of the re-securitization's layers: Step 2 of Section 25.7 with a piece in place of a loan."""
    out = np.zeros(len(layers))
    L = np.arange(20 * N_DEALS + 1) * UNITS / N_DEALS      # the new pool's loss, per unit of its notional
    for market in np.arange(-lim, lim + 1e-9, h):
        dist = convolve_power(piece_dist_given_market(market, rho_deal), N_DEALS)
        out += norm.pdf(market) * h * np.array([dist @ tranche_loss(L, a, b) for a, b in layers])
    return out


book = {0.10: (1.08, 44.13, 94.36, 99.96), 0.30: (3.83, 40.89, 80.19, 97.71),
        0.60: (7.55, 36.64, 63.34, 86.84), 0.90: (11.25, 32.18, 49.18, 69.72)}
print("%14s %14s %13s %9s %13s %14s" % ("inter-deal rho", "AAA 30-100%", "AA 12-30%", "3-12%", "equity 0-3%", "weighted sum"))
aaa = {}
for rho_deal in sorted(book):
    el = 100 * cdo_squared(rho_deal)
    aaa[rho_deal] = el[0]
    weighted = 0.70 * el[0] + 0.18 * el[1] + 0.09 * el[2] + 0.03 * el[3]
    print("%14.2f %13.2f%% %12.2f%% %8.2f%% %12.2f%% %13.2f%%" % (rho_deal, *el, weighted))
    for got, want in zip(el, book[rho_deal]):
        assert abs(got - want) < 6e-3, rho_deal
print()
show("every row adds to the mezzanine piece's own loss, %", weighted, 20.19, tol=6e-3)
show("AAA loss, inter-deal rho 0.90 over 0.10", aaa[0.90] / aaa[0.10], 10.5, tol=0.02)

inter-deal rho    AAA 30-100%     AA 12-30%     3-12%   equity 0-3%   weighted sum
          0.10          1.08%        44.13%    94.36%        99.96%         20.19%


          0.30          3.83%        40.89%    80.19%        97.71%         20.19%
          0.60          7.55%        36.64%    63.34%        86.84%         20.19%
          0.90         11.25%        32.18%    49.18%        69.72%         20.19%

every row adds to the mezzanine piece's own loss, %      20.1870   book     20.1900   ok
AAA loss, inter-deal rho 0.90 over 0.10                 10.4605   book     10.5000   ok


## 5. The test that deleverages the deal automatically — Example 26.5

A \$400 million CLO with \$256 million of AAA notes and a 120% senior
overcollateralization trigger. The cure works on the denominator: paying down the notes
shrinks the base the ratio is measured against, so \$6 million cures a breach that
looked like 2.8 points — and the cure gets rapidly more expensive as par erodes.

In [9]:
collateral, aaa_notes, trigger = 400.0, 256.0, 1.20

show("opening OC ratio, %", 100 * collateral / aaa_notes, 156.25)
show("collateral par at which the test fails", trigger * aaa_notes, 307.2)
show("erosion the cushion absorbs", collateral - trigger * aaa_notes, 92.8)
show("that erosion as a fraction of the pool, %", 100 * (collateral - trigger * aaa_notes) / collateral, 23.2, tol=0.01)
print()
for par, ratio_book, cure_book in ((300.0, 117.2, 6.0), (280.0, 109.4, 22.7)):
    cure = aaa_notes - par / trigger                       # X such that par / (notes - X) = trigger
    show("par %.0f: OC ratio, %%" % par, 100 * par / aaa_notes, ratio_book, tol=0.01)
    show("par %.0f: pay-down that restores the test" % par, cure, cure_book, tol=0.01)
    show("par %.0f: as a fraction of the deal, %%" % par, 100 * cure / collateral, 100 * cure_book / collateral, tol=0.01)

opening OC ratio, %                                    156.2500   book    156.2500   ok
collateral par at which the test fails                 307.2000   book    307.2000   ok
erosion the cushion absorbs                             92.8000   book     92.8000   ok
that erosion as a fraction of the pool, %               23.2000   book     23.2000   ok

par 300: OC ratio, %                                   117.1875   book    117.2000   ok
par 300: pay-down that restores the test                 6.0000   book      6.0000   ok
par 300: as a fraction of the deal, %                    1.5000   book      1.5000   ok
par 280: OC ratio, %                                   109.3750   book    109.4000   ok
par 280: pay-down that restores the test                22.6667   book     22.7000   ok
par 280: as a fraction of the deal, %                    5.6667   book      5.6750   ok


## 6. Why a senior claim recovers less without the borrower getting worse — Example 26.6

Three capital structures on one company, same enterprise value and same total debt, differing only in
how much junior debt sits beneath the first lien. A first lien is impaired when enterprise value falls
below **its own** par, so what protects it is everything beneath it, junior debt and equity alike, measured
against its own size. With total debt fixed, shrinking the junior debt makes the first lien the whole of the debt.

Move the split and the enterprise-value decline that first touches the first lien moves from 66.7% to
16.7%, and one distress outcome pays the first lien 100% or 70% with the borrower unchanged. Then two
mechanisms compound on top: covenant-lite defers the event, and a liability-management exercise moves
collateral out of the lien.

The parameters are worth moving. Raising the junior debt restores the traditional recovery, which is the
point: the statistic is a fact about capital structures, not about credit quality.

In [10]:
EV0, TOTAL_DEBT, EV_DISTRESS = 1200.0, 1000.0, 700.0
STRUCTS = [("traditional", 400.0, 600.0), ("intermediate", 700.0, 300.0),
           ("loan-only", 1000.0, 0.0)]

print("Total debt is %.0f in every structure; only the split differs." % TOTAL_DEBT)
print("\n%-16s %10s %10s %26s" % ("structure", "1L par", "junior", "EV decline that impairs 1L"))
for name, first, junior in STRUCTS:
    show("%-12s decline that impairs the 1L, %%" % name,
         100.0 * (1.0 - first / EV0),
         {"traditional": 66.7, "intermediate": 41.7, "loan-only": 16.7}[name], tol=0.01)
    print("%-16s %10.0f %10.0f %25.1f%%" % (name, first, junior, 100.0 * (1.0 - first / EV0)))

print("\nOne distress outcome, enterprise value %.0f, applied to all three:" % EV_DISTRESS)
print("%-16s %18s %18s" % ("structure", "1L recovery", "junior recovery"))
for name, first, junior in STRUCTS:
    r1 = 100.0 * min(EV_DISTRESS, first) / first
    show("%-12s first-lien recovery, %%" % name, r1,
         {"traditional": 100.0, "intermediate": 100.0, "loan-only": 70.0}[name])
    rj = 100.0 * max(EV_DISTRESS - first, 0.0) / junior if junior else float("nan")
    print("%-16s %17.1f%% %17s" % (name, r1, ("%.1f%%" % rj) if junior else "--"))

print()
print("The same enterprise outcome pays the first lien 100.0%% or %.1f%%, and the borrower"
      % (100.0 * min(EV_DISTRESS, 1000.0) / 1000.0))
print("is identical in all three columns. An average first-lien recovery taken across a")
print("market that shifted from the first row to the third falls without any borrower")
print("getting worse: the statistic moved because the composition moved.")

# The two mechanisms that compound on top, each a further reduction in what reaches the lien.
print("\n%-42s %12s" % ("loan-only structure, cumulative", "1L recovery"))
chain = [("as above", 1.0), ("covenant-lite defers the event, -10% EV", 0.90),
         ("an LME moves 20% of value out of the lien", 0.72)]
for label, factor in chain:
    rec = 100.0 * min(EV_DISTRESS * factor, 1000.0) / 1000.0
    print("%-42s %11.1f%%" % (label, rec))
show("covenant-lite alone, recovery %", 100.0 * min(EV_DISTRESS * 0.90, 1000.0) / 1000.0, 63.0)
show("and the LME on top, recovery %", 100.0 * min(EV_DISTRESS * 0.72, 1000.0) / 1000.0, 50.4)
print()
print("Senior, secured, first in the waterfall -- and recovering half, with none of the")
print("three words having become false. Recovery is a documentation variable here, not a")
print("credit variable, which is why Section 24.4 distrusts the 40% convention.")

Total debt is 1000 in every structure; only the split differs.

structure            1L par     junior EV decline that impairs 1L
traditional  decline that impairs the 1L, %             66.6667   book     66.7000   ok
traditional             400        600                      66.7%
intermediate decline that impairs the 1L, %             41.6667   book     41.7000   ok
intermediate            700        300                      41.7%
loan-only    decline that impairs the 1L, %             16.6667   book     16.7000   ok
loan-only              1000          0                      16.7%

One distress outcome, enterprise value 700, applied to all three:
structure               1L recovery    junior recovery
traditional  first-lien recovery, %                    100.0000   book    100.0000   ok
traditional                  100.0%             50.0%
intermediate first-lien recovery, %                    100.0000   book    100.0000   ok
intermediate                 100.0%              0.0%
lo